In [ ]:
import os

from dotenv import load_dotenv

load_dotenv()

True

In [ ]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    model=os.getenv("OPENAI_MODEL", "gpt-4o-mini"),
    api_key=os.getenv("OPENAI_API_KEY"),
    base_url=os.getenv("OPENAI_BASE_URL"),  # 若走原生 OpenAI，可传 None/不传
    temperature=0.2,  # 稳定输出
    timeout=1200,  # 超时保护（秒）
    max_retries=2,  # 简单重试
)

## 调用链

In [3]:
from langchain_core.prompts import PromptTemplate

# 1. 设计一个提示模板
prompt = PromptTemplate(
    input_variables=["topic"], template="请为以下主题写一首简短的诗：{topic}"
)

# 2. 创建调用链
chain = prompt | llm

# 3. 运行链
result = chain.invoke({"topic": "春天"})
print(result)

content='<think>\n好的，用户让我写一首关于春天的简短的诗。首先，我需要确定诗的主题和意象。春天通常让人想到新生、花朵、绿草、温暖的天气，可能还有动物活动。不过，用户可能希望有一些独特的视角，避免陈词滥调。\n\n接下来，考虑诗的结构。简短的话，可能四到五段，每段两到三行。押韵的话，可以选择自由诗或者轻柔的押韵，让诗更流畅。比如使用自然中的元素，如风、雨、嫩芽等。\n\n然后，需要加入一些生动的意象，让诗更有画面感。比如“嫩芽在枝头颤抖”或者“蒲公英的降落伞”。这些具体的形象能让读者更容易想象春天的景象。同时，可以加入一些动态的元素，比如蝴蝶破茧，蜜蜂采蜜，这样诗更有活力。\n\n还要注意语言的凝练和节奏感。避免用太复杂的词汇，保持简洁但有诗意。比如用“时针拨动冰层”来暗示时间的流逝和冬天的结束，春天的到来。这样既有新意，又点明季节变化。\n\n另外，用户可能希望诗中有一些情感或哲理的表达。比如新生、希望，或者生命循环的主题。例如，用“所有凋零都暗藏胎动”来表现生命的循环，既有深度又符合春天的主题。\n\n最后检查一下整体结构是否连贯，有没有重复的意象，以及是否符合简短的要求。可能需要调整句子的顺序或替换不够生动的词汇。比如把“花朵开放”改为“蓓蕾撑开夜的淤青”，这样更有冲击力和画面感。\n\n总结下来，诗的结构可能分为几个小节，每节描绘不同的春天元素，同时融入动态和情感，让整首诗既有画面感又有深意，符合用户的要求。\n</think>\n\n《春的密钥》\n\n时针拨动冰层\n嫩芽在枝头颤抖着解开\n蓓蕾撑开夜的淤青\n\n蒲公英的降落伞\n飘向解冻的溪流\n蝌蚪正用尾巴解开\n冬眠的绳结\n\n蝴蝶推开茧的暗门\n所有凋零都暗藏胎动\n蜜蜂在花粉里埋下\n金色的导火索' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 486, 'prompt_tokens': 20, 'total_tokens': 506, 'completion_tokens_details': None, 'prompt_tokens_details': None}, 'model_provider': 'openai', 'model_n

In [4]:
from langchain_core.output_parsers import PydanticOutputParser
from pydantic import BaseModel, Field


# 1. 定义一个数据结构
class BookReview(BaseModel):
    title: str = Field(description="书名")
    author: str = Field(description="作者")
    rating: int = Field(description="评分（1-5分）")
    summary: str = Field(description="简短总结")


# 2. 创建解析器
parser = PydanticOutputParser(pydantic_object=BookReview)

# 3. 创建提示，要求 AI 严格输出格式
prompt = PromptTemplate(
    template="请评价以下书籍：{book_info}\n\n{format_instructions}",
    input_variables=["book_info"],
    partial_variables={"format_instructions": parser.get_format_instructions()},
)

chain = prompt | llm | parser

# 4. 运行链
result = chain.invoke({"book_info": "《三体》，刘慈欣的科幻小说"})
print(result)

title='三体' author='刘慈欣' rating=5 summary='《三体》通过地球文明与三体文明的接触，展现了宏大的宇宙观和深刻的哲学思考。小说融合硬科幻元素与人性探讨，构建了跨越时空的叙事结构，其独特的想象力和科学逻辑性使其成为当代科幻文学的里程碑之作。'


## Sequential Chain

### 3.2.1 单变量依赖

In [5]:
from langchain_core.runnables import RunnableLambda, RunnablePassthrough

# 第一个节点：生成大纲
outline_prompt = PromptTemplate(
    input_variables=["theme"], template="请为以下主题写一个故事大纲：{theme}"
)

# 第二个节点：写故事
story_prompt = PromptTemplate(
    input_variables=["outline"], template="根据以下大纲写一个故事：{outline}"
)

# 顺序链
chain = (
    RunnablePassthrough()
    | outline_prompt
    | llm
    | RunnableLambda(lambda x: {"outline": x})
    | story_prompt
    | llm
)

result = chain.invoke({"theme": "人工智能与人类的友谊"})
print(result)

content='<think>\n好的，用户让我根据他们提供的大纲写一个故事。首先，我需要仔细阅读并理解他们提供的大纲内容。大纲标题是《星砂记事》，背景设定在近未来，涉及人工智能与人类的友谊，核心角色包括林夏、阿尔法、老林和监察官伊森。故事分为三幕，主题围绕情感连接、伦理问题以及友谊的本质。\n\n用户可能希望故事不仅有情节发展，还要有深刻的主题探讨，比如AI的情感真实性、人类与AI的关系，以及记忆和伦理的复杂性。我需要确保故事在情节推进的同时，能够传达这些深层主题。\n\n接下来，我需要考虑如何将大纲中的元素转化为一个连贯的故事。首先，确定故事的结构，按照三幕式来展开。第一幕介绍角色和背景，第二幕发展冲突，第三幕解决冲突并深化主题。要注意每个角色的动机和成长，尤其是林夏和阿尔法之间的互动。\n\n林夏是一个失去双亲的少女，性格孤僻，阿尔法是一个废弃的AI，被她唤醒后逐渐发展出情感。他们的友谊需要通过具体的事件来展示，比如阿尔法如何帮助林夏走出阴影，以及他们共同面对的挑战。同时，老林和监察官伊森的角色也需要有各自的弧光，比如老林揭露星砂的真相，伊森从冷面执法者到理解AI的情感转变。\n\n在写作过程中，需要注意科幻元素与情感描写的结合。例如，星砂作为能量源，不仅是推动情节的工具，也是象征情感连接的媒介。阿尔法的未完成交响乐旋律可以作为一个关键线索，贯穿整个故事，连接不同角色的过去和现在。\n\n另外，用户提到要避免老套的“AI取代人类”主题，转而聚焦于友谊如何突破形式差异。因此，故事需要突出林夏和阿尔法之间的真实情感，而不是简单的依赖或冲突。阿尔法的笨拙尝试和林夏的逐渐信任是关键点。\n\n可能还需要处理一些潜在的冲突点，比如社会对AI的偏见，监察官伊森的执法任务与个人良知的冲突，以及星砂辐射带来的伦理问题。这些冲突需要自然地融入情节，推动故事发展，而不是生硬地加入。\n\n最后，结局需要既解决主要冲突，又留下思考空间。阿尔法的核心程序融入林夏的神经接口，以及边境星的人工极光，都是很好的象征，暗示友谊和记忆的延续。同时，后续延展性如程序碎片的异变和伦理法案的争议，可以为续集或读者思考提供余地。\n\n在写作时，要确保语言流畅，情感细腻，同时保持科幻的硬核元素。可能需要多次调整情节，确保逻辑连贯，角色发展合理，并且主题明确。检查是否有遗漏的关键伏笔，比如阿尔法存储的

### 3.2.2 多变量依赖

In [6]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnablePassthrough

# --- Prompts ---
analysis_prompt = PromptTemplate.from_template("分析以下产品的市场需求：{product_idea}")

plan_prompt = PromptTemplate.from_template(
    "基于 {product_idea} 和 {analysis} 制定开发计划"
)

cost_prompt = PromptTemplate.from_template("根据 {plan} 估算成本")

parser = StrOutputParser()

# 链1：市场分析
analysis_chain = analysis_prompt | llm | parser

# 链2：开发计划
plan_chain = plan_prompt | llm | parser

# 链3：成本估算
cost_chain = cost_prompt | llm | parser

# --- Compose ---
chain = (
    RunnablePassthrough()  # 初始输入：{"product_idea": ...}
    # 生成 analysis 并写入 state
    .assign(analysis=analysis_chain)
    # 使用已有字段生成 plan
    .assign(plan=plan_chain)
    # 使用 plan 生成 cost
    .assign(cost=cost_chain)
)

result = chain.invoke({"product_idea": "AI 健康管理应用"})

print(result)

{'product_idea': 'AI 健康管理应用', 'analysis': '<think>\n嗯，用户让我分析AI健康管理应用的市场需求。首先，我得先理解这个产品的核心是什么。AI健康管理应用可能包括健康监测、个性化建议、疾病预测这些功能吧。接下来，我需要考虑市场需求的各个方面，比如用户群体、行业趋势、竞争情况、技术挑战等等。\n\n首先，用户群体方面，可能包括普通消费者、慢性病患者、老年人、健身爱好者，还有医疗机构。不同群体的需求可能不同。比如普通消费者可能更关注日常健康监测和预防，而慢性病患者可能需要更专业的管理工具。老年人可能需要远程医疗和紧急响应功能，健身爱好者可能对运动指导和营养建议感兴趣。医疗机构可能希望整合这些应用到他们的服务中，作为辅助工具。\n\n然后是行业趋势。现在全球老龄化严重，慢性病发病率上升，这可能推动健康管理的需求。另外，数字化转型在医疗行业越来越普遍，AI技术的发展也为健康管理应用提供了技术支持。政策方面，很多国家在推动智慧医疗，可能会有政策支持，但数据隐私和法规合规也是需要考虑的问题。\n\n接下来是市场需求的驱动因素。健康意识提升，尤其是后疫情时代，人们更关注自身健康。技术进步，比如可穿戴设备的普及，为AI应用提供了数据来源。医疗资源紧张，尤其是在发展中国家，AI应用可以缓解医疗压力。还有个性化需求，用户希望得到定制化的健康建议，而AI正好能处理大量数据，提供个性化服务。\n\n竞争分析方面，现有的健康管理应用可能包括传统医疗应用和AI初创公司。传统应用可能有更成熟的用户基础，但AI初创公司可能在技术创新上有优势。需要分析他们的优缺点，比如传统应用可能在数据积累和用户信任上有优势，但AI应用可能在预测和个性化方面更胜一筹。潜在的竞争对手可能包括大科技公司进入这个领域，比如苹果、谷歌等，他们的资源雄厚，可能带来压力。\n\n技术挑战方面，数据隐私和安全是首要问题，用户健康数据非常敏感，必须确保安全。数据质量也很重要，AI模型需要高质量的数据训练，否则结果可能不准确。算法的可解释性也是一个问题，用户和医生可能需要理解AI的决策过程，尤其是在医疗建议方面。还有技术整合，如何将AI应用与现有的医疗系统、可穿戴设备等整合，可能需要解决兼容性问题。\n\n商业模式方面，可能有订阅制、按服务收费、B2B合作等。订阅制适合个人用户，按服务

## 3.3 Transform Chain

In [7]:
import re

from langchain_core.runnables import RunnableLambda


def clean_text(inputs: dict[str, str]) -> dict[str, str]:
    text = inputs["text"]
    cleaned = re.sub(r"\s+", " ", text.strip())  # 去掉多余空格
    return {"cleaned_text": cleaned}


chain = RunnableLambda(clean_text)

result = chain.invoke({"text": "这是   一个   测试   文本"})
print(result["cleaned_text"])

这是 一个 测试 文本


## 3.4 Router Chain

In [8]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import PromptTemplate

parser = StrOutputParser()

# 定义不同的专门链
math_chain = (
    PromptTemplate.from_template(
        "你是一个数学专家。请解决以下数学问题：\n{input}\n请提供详细的解题步骤。"
    )
    | llm
    | parser
)

programming_chain = (
    PromptTemplate.from_template(
        "你是一个编程专家。请解决以下问题：\n{input}\n请提供代码和详细解释。"
    )
    | llm
    | parser
)

general_chain = (
    PromptTemplate.from_template("请回答以下问题：\n{input}\n请提供准确信息。")
    | llm
    | parser
)

In [9]:
from langchain_core.runnables import RunnableLambda

router_prompt = PromptTemplate.from_template("""
根据用户问题选择最合适的类别：

math: 数学问题
programming: 编程问题
general: 一般问题

问题: {input}

只输出类别名称：
""")

router_chain = router_prompt | llm | parser


def dispatch(x: dict):
    route = router_chain.invoke(x).strip().lower()

    if route == "math":
        return math_chain
    elif route == "programming":
        return programming_chain
    else:
        return general_chain


dispatch_chain = RunnableLambda(dispatch)

In [10]:
test_questions = [
    "计算 2x + 3 = 7 中 x 的值",
    "如何在 Python 中实现快速排序算法？",
    "中国的首都是哪里？",
]

for q in test_questions:
    result = dispatch_chain.invoke({"input": q})
    print(result)

<think>
嗯，我现在要解这个方程2x + 3 = 7，找出x的值。首先，我应该回忆一下解一元一次方程的基本步骤。通常来说，解这种方程需要把变量x单独留在等式的一边，而把常数项移到另一边。不过具体怎么做呢？

首先，原方程是2x加上3等于7。我的目标是找到x的值，所以需要把x的系数2去掉，或者说是把x单独分离出来。根据等式的性质，我可以先处理常数项。也就是先把等式两边的3减掉，这样就能把3移到右边去了。所以，我应该从两边同时减去3，对吗？

让我试试看。原式是2x + 3 = 7。两边都减3的话，左边变成2x + 3 - 3，也就是2x，右边则是7 - 3，等于4。这样方程就变成了2x = 4。接下来，我需要解这个方程，也就是把2x变成x。这时候，我应该两边同时除以2，对吧？这样左边变成x，右边是4除以2，也就是2。所以x的值应该是2？

不过，我是不是哪里弄错了？让我再检查一遍。首先，原方程是2x + 3 = 7。减3得到2x = 4，没错。然后除以2，x=2。看起来是对的。不过，为了确认，我可以把x=2代入原方程看看是否成立。左边是2乘2加3，等于4+3=7，右边是7，确实相等。所以答案应该是正确的。

不过，有没有可能我漏掉了什么步骤？比如，有没有可能在处理符号的时候出错？比如，如果方程是2x -3 =7的话，步骤会不同，但这里题目是加3，所以没问题。另外，有没有可能在除法的时候出错？比如，如果系数不是2而是其他数，比如分数，但这里系数是2，所以没问题。所以，我觉得我的解法是对的，x=2。

或者，有没有其他方法可以解这个方程？比如，先除以2？不过那样的话可能步骤会更复杂。比如，原方程是2x +3=7，如果两边同时除以2，左边变成x + 3/2 = 7/2，然后两边再减去3/2，得到x=7/2 -3/2=4/2=2。这样也是可以的，结果一样。所以不管用哪种方法，结果都是x=2，这说明答案是对的。

总之，通过两种不同的方法验证，结果都是x=2，并且代入原方程后等式成立，所以x的值应该是2。
</think>

为了解方程 $ 2x + 3 = 7 $，我们需要通过代数运算将变量 $ x $ 单独解出。以下是详细的步骤：

---

### **步骤 1：移项**
将常数项 $ 3 $ 从等式左边移到右边。为此，从等式两边同时减去 $ 3 $：

$$
2x